# VLM Defect Detection — LLaVA Fine-tuning on MVTec-AD

**Runtime:** Google Colab Pro · A100 (40 GB) · High-RAM  
**Estimated training time:** ~45 min for 3 epochs on 5,354 samples

## How this notebook is organized
| Section | What it does |
|---|---|
| 1 · Runtime check | Verify GPU + VRAM |
| 2 · Mount Drive | Single mount; all outputs go to Drive |
| 3 · Install deps | One-time install, cached in Drive |
| 4 · Dataset | Download MVTec-AD via Kaggle API |
| 5 · Prepare JSON | Convert raw images → training JSON |
| 6 · Train | QLoRA fine-tuning with progress logging |
| 7 · Evaluate | Accuracy + F1 on held-out val split |
| 8 · Inference demo | Test on custom images |

> **Checkpoint resume:** If Colab disconnects, re-run sections 1-3, then jump to section 6 — training will resume from the last checkpoint automatically.

## 1 · Runtime Check

In [ ]:
import subprocess, sys

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout
print(gpu_info)

# Warn if not on A100
if 'A100' not in gpu_info and 'L4' not in gpu_info:
    print('WARNING: Not on A100/L4. Switch runtime: Runtime → Change runtime type → A100.')
    print('Training will still work on T4 but will be ~4x slower.')
else:
    print('A100/L4 detected — proceeding with full-precision LoRA config.')

import torch
print(f'PyTorch: {torch.__version__}  |  CUDA: {torch.version.cuda}')
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {vram_gb:.1f} GB')

## 2 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
BASE_DIR = '/content/drive/MyDrive/vlm-defect'
DATASET_DIR = f'{BASE_DIR}/mvtec_anomaly_detection'
DATA_JSON   = f'{BASE_DIR}/data/mvtec_train.json'
CKPT_DIR    = f'{BASE_DIR}/checkpoints/llava-mvtec-lora'
REPO_DIR    = '/content/vlm-defect-detection'   # cloned locally (fast /content SSD)

os.makedirs(f'{BASE_DIR}/data', exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
print('Drive mounted. Base dir:', BASE_DIR)

## 3 · Install Dependencies

In [ ]:
# Clone the repo to /content (fast SSD, not Drive)
import os, subprocess

if not os.path.exists(REPO_DIR):
    subprocess.run(
        ['git', 'clone', 'https://github.com/Pranavk098/vlm-defect-detection', REPO_DIR],
        check=True
    )
    print('Repo cloned to', REPO_DIR)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Repo updated')

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

In [ ]:
# Install training deps — uses pyproject.toml [train] extras
# bitsandbytes + peft + trl + accelerate + wandb
!pip install -q -e '.[train,colab]'

# Flash Attention 2 — pre-compiled wheel for Colab A100 CUDA 12.x
# This saves ~30% VRAM and speeds up training
!pip install -q flash-attn --no-build-isolation

print('All dependencies installed.')

In [ ]:
# Quick sanity check
import torch
import transformers, peft, trl, bitsandbytes

print(f'transformers : {transformers.__version__}')
print(f'peft         : {peft.__version__}')
print(f'trl          : {trl.__version__}')
print(f'bitsandbytes : {bitsandbytes.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

### 3b · (Optional) W&B Login — survives Colab disconnects

In [ ]:
# W&B lets you monitor training from your phone even after Colab disconnects.
# Skip this cell if you don't want W&B.
USE_WANDB = False   # set True if you want W&B

if USE_WANDB:
    import wandb
    wandb.login()   # paste your API key when prompted
    print('W&B authenticated.')

## 4 · Download MVTec-AD Dataset

In [ ]:
import os

if os.path.isdir(DATASET_DIR) and len(os.listdir(DATASET_DIR)) > 5:
    print(f'Dataset already present at {DATASET_DIR} — skipping download.')
else:
    print('Downloading MVTec-AD via Kaggle API...')
    # Option A: Kaggle (recommended — fastest)
    # 1. Go to https://www.kaggle.com/settings/account → Create API token → download kaggle.json
    # 2. Upload kaggle.json to your Drive: MyDrive/vlm-defect/kaggle.json
    # 3. Run this cell

    kaggle_json = f'{BASE_DIR}/kaggle.json'
    if os.path.exists(kaggle_json):
        os.makedirs(os.path.expanduser('~/.config/kaggle'), exist_ok=True)
        os.system(f'cp {kaggle_json} ~/.config/kaggle/kaggle.json && chmod 600 ~/.config/kaggle/kaggle.json')
        os.makedirs(DATASET_DIR, exist_ok=True)
        os.system(f'kaggle datasets download -d ipythonx/mvtec-ad --path {BASE_DIR} --unzip')
        print(f'Downloaded to {DATASET_DIR}')
    else:
        print(f'kaggle.json not found at {kaggle_json}.')
        print('Manual option: upload kaggle.json to MyDrive/vlm-defect/ and re-run.')
        print('Or: download MVTec-AD manually from https://www.mvtec.com/company/research/datasets/mvtec-ad')
        print('    and place it at:', DATASET_DIR)

## 5 · Prepare Training JSON

In [ ]:
import os

if os.path.exists(DATA_JSON):
    import json
    with open(DATA_JSON) as f:
        n = len(json.load(f))
    print(f'JSON already exists: {DATA_JSON} ({n:,} records) — skipping.')
else:
    os.chdir(REPO_DIR)
    !python scripts/prepare_data.py --root {DATASET_DIR} --out {DATA_JSON}
    print('JSON prepared.')

## 6 · Fine-tune LLaVA (QLoRA on A100)

In [ ]:
# Load and patch the A100 config with the actual Drive paths
import sys
sys.path.insert(0, REPO_DIR)

from src.vlm_defect.config import load_config

cfg = load_config(f'{REPO_DIR}/configs/colab_a100.yaml')

# Patch paths to point at Drive
cfg.data.dataset_root = DATASET_DIR
cfg.data.json_path    = DATA_JSON
cfg.training.output_dir = CKPT_DIR

# Flash Attention 2 is available on Colab A100 after the pip install above
cfg.model.attn_implementation = 'flash_attention_2'

# W&B
cfg.training.report_to = 'wandb' if USE_WANDB else 'none'

print('Config summary:')
print(f'  Model       : {cfg.model.base_id}')
print(f'  LoRA rank   : {cfg.lora.r}  alpha: {cfg.lora.alpha}')
print(f'  Quantization: 4-bit={cfg.quantization.load_in_4bit}')
print(f'  Batch       : {cfg.training.batch_size} × {cfg.training.grad_accumulation} accum = {cfg.training.batch_size * cfg.training.grad_accumulation} effective')
print(f'  Epochs      : {cfg.training.epochs}')
print(f'  Output dir  : {cfg.training.output_dir}')
print(f'  Report to   : {cfg.training.report_to}')

In [ ]:
# This cell starts training. If Colab disconnects and you re-run:
# - The HuggingFace Trainer will find the latest checkpoint in CKPT_DIR
#   and resume automatically (resume_from_checkpoint=True is the default).

from src.vlm_defect.trainer import run_training

run_training(cfg)
print('Training complete. Checkpoint at:', CKPT_DIR)

## 7 · Evaluate on Validation Split

In [ ]:
from src.vlm_defect.model import load_inference_model
from src.vlm_defect.evaluate import evaluate

model, processor = load_inference_model(CKPT_DIR, cfg)

results = evaluate(
    model, processor,
    json_path=DATA_JSON,
    dataset_root=DATASET_DIR,
    cfg=cfg,
    batch_size=8,
)

print(f'\nAccuracy: {results["accuracy"]:.4f}')
print(results['report'])

## 8 · Inference Demo

In [ ]:
# Upload an image and run the model on it
from google.colab import files
from PIL import Image
import io

uploaded = files.upload()
for fname, data in uploaded.items():
    image = Image.open(io.BytesIO(data)).convert('RGB')
    break

display(image)

In [ ]:
import torch

QUESTION = (
    'Is there any anomaly in this image? '
    'If yes, answer Yes and describe the defect briefly. '
    'If no, just answer No.'
)

conversation = [
    {'role': 'user', 'content': [
        {'type': 'image'},
        {'type': 'text', 'text': QUESTION},
    ]}
]

text = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
inputs = processor(text=text, images=image, return_tensors='pt').to(model.device)

with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False)

answer = processor.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('Model answer:', answer)